# Bronze Layer — Source Blob config/ → Bronze Volume (XML Reference Data)

**Day 6 | One-time load — static reference data, no partitioning.**

Copies all 19 XML reference files from the `config/` folder in the instructor source blob into the Bronze Volume. These are static lookup tables (connector types, tariffs, firmware manifests, station configs by state) — they rarely change, so a one-time full copy is the right pattern. Re-run only when the source is updated.

---

### Source layout

```
wasbs://source@dataenggdailystorage.blob.core.windows.net/
  └── config/          ← flat folder, no subfolders
        ├── connector_standards.xml
        ├── connector_types.xml
        ├── firmware_manifest_v7_4_2.xml
        ├── firmware_manifest_v7_5_0.xml
        ├── firmware_manifest_v8_0_1.xml
        ├── network_topology.xml
        ├── operator_registry.xml
        ├── site_types.xml
        ├── states.xml
        ├── station_config_ACT.xml
        ├── station_config_NSW.xml
        ├── station_config_NT.xml
        ├── station_config_QLD.xml
        ├── station_config_SA.xml
        ├── station_config_TAS.xml
        ├── station_config_VIC.xml
        ├── station_config_WA.xml
        ├── tariff_rate_card_202606.xml
        └── tariffs.xml
```

### Bronze Volume target

```
/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/
  └── config/
        └── *.xml   (19 files — same names as source, flat folder)
```

---

**When to re-run:** Only when source XML files are updated by the instructor. For daily ingestion pipelines these files are broadcast as lookup tables from Bronze — no need to reload them every day.

## Cell 1 — Authenticate to source blob storage

In [ ]:
SCOPE = "kv-ev-scope"

STORAGE_ACCOUNT = dbutils.secrets.get(scope=SCOPE, key="source-storage-account")
CONTAINER       = dbutils.secrets.get(scope=SCOPE, key="source-container")
SAS_TOKEN       = dbutils.secrets.get(scope=SCOPE, key="source-sas-token")

spark.conf.set(
    f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SAS_TOKEN
)

SOURCE_ROOT = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"

print(f"Storage account : {STORAGE_ACCOUNT}")
print(f"Container       : {CONTAINER}")
print(f"Source root     : {SOURCE_ROOT}")
print("Source blob authenticated — OK")

## Cell 2 — Set paths

No parameters to change. Config is a flat folder — full copy every time.

In [ ]:
SOURCE_PATH  = f"{SOURCE_ROOT}/config/"
BRONZE_PATH  = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/config/"

print(f"Source : {SOURCE_PATH}")
print(f"Bronze : {BRONZE_PATH}")

## Cell 3 — List source XML files

In [ ]:
try:
    source_files = dbutils.fs.ls(SOURCE_PATH)
except Exception as e:
    raise Exception(f"Cannot list source config/ folder: {e}")

# config/ is flat — no subdirectories
xml_files = [f for f in source_files if f.name.endswith(".xml")]

print(f"XML files found: {len(xml_files)}")
for f in xml_files:
    print(f"  {f.name:<55}  [{round(f.size/1024, 1)} KB]")

## Cell 4 — Copy XML files to Bronze Volume

In [ ]:
copied  = []
skipped = []

for file_info in xml_files:
    dest_path = BRONZE_PATH + file_info.name
    try:
        dbutils.fs.cp(file_info.path, dest_path)
        copied.append(dest_path)
        print(f"  COPIED  {file_info.name}")
    except Exception as e:
        skipped.append((file_info.path, str(e)))
        print(f"  FAILED  {file_info.name} — {e}")

print(f"\nResult: {len(copied)} copied, {len(skipped)} failed")
if skipped:
    raise Exception(f"{len(skipped)} file(s) failed — check output above.")

## Cell 5 — Verify files in Bronze Volume

In [ ]:
try:
    bronze_files = dbutils.fs.ls(BRONZE_PATH)
except Exception as e:
    raise Exception(f"Cannot list Bronze config/ folder: {e}")

bronze_xml = [f for f in bronze_files if f.name.endswith(".xml")]

status = "PASS" if len(bronze_xml) == len(xml_files) else "FAIL"
print(f"[{status}] Source: {len(xml_files)} files  →  Bronze: {len(bronze_xml)} files")
for f in bronze_xml:
    print(f"  {f.name}")

assert len(bronze_xml) == len(xml_files), (
    f"Count mismatch — source: {len(xml_files)}, bronze: {len(bronze_xml)}"
)
print("\nVerification passed — all XML files confirmed in Bronze Volume.")

## Cell 6 — Read one XML file to confirm it is parseable

Spark's native XML reader requires the `com.databricks.spark.xml` package (available by default on Databricks Runtime 13+). If the cluster doesn't have it, the `spark.read.format("xml")` call will fail — install the Maven package `com.databricks:spark-xml_2.12:0.18.0` on the cluster.

For quick Bronze validation, `dbutils.fs.head` is used here — no Spark XML needed.

In [ ]:
if bronze_xml:
    sample = bronze_xml[0].path
    print(f"First 500 bytes of: {bronze_xml[0].name}")
    print("-" * 60)
    print(dbutils.fs.head(sample, 500))
    print("-" * 60)
    print(f"\nAll {len(bronze_xml)} XML files are in Bronze Volume and readable.")
    print("Silver layer can load these with spark.read.format('xml').option('rowTag', '<tag>').load(path)")